In [ ]:
import numpy as np
from astropy.io import fits
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from astropy import units as u
from astropy.coordinates import SkyCoord

sns.set_theme(style="darkgrid")

# Specify the path to your FITS file
file_path = '/lustre/work/client/users/cdcook/fits_structs/000906_sky0001_1a_match.fit'
file_path = '/work/group/astro/rotse/data/3b/13/08/01/prod/130801_vsp2218+4034_3b_match.fit'

In [ ]:
# Function to extract and print Min/Max RA and Dec
def get_min_max_ra_dec(data):
    if 'RA' in data.columns.names and 'DEC' in data.columns.names:
        ra = data['RA']
        dec = data['DEC']
        print(f"Min RA: {np.min(ra)}")
        print(f"Max RA: {np.max(ra)}")
        print(f"Min Dec: {np.min(dec)}")
        print(f"Max Dec: {np.max(dec)}")
    else:
        print("RA or DEC columns not found in the data.")

# Function to count the number of exposures and objects based on magnitudes array
def count_exposures_objects(data):
    if 'M' in data.columns.names:
        magnitudes = data['M']
        num_objects = magnitudes.shape[0]  # Number of objects
        num_exposures = magnitudes.shape[1]  # Number of exposures
        print(f"Number of Exposures: {num_exposures}")
        print(f"Number of Objects: {num_objects}")
        return num_exposures, num_objects
    else:
        print("M column not found in the data.")
        return 0, 0

# Function to create a data structure for RA, Dec, Magnitude, Flags, and Julian Date
def create_data_structure(data):
    required_columns = ['RA', 'DEC', 'M', 'FLAGS', 'RFLAGS', 'JD']
    missing_columns = [col for col in required_columns if col not in data.columns.names]
    
    if missing_columns:
        print(f"Missing columns: {', '.join(missing_columns)}")
        return []

    ra_values = data['RA']
    dec_values = data['DEC']
    magnitudes = data['M']
    errors = data['ERR'] if 'ERR' in data.columns.names else np.full_like(magnitudes, np.nan)
    flags = data['FLAGS'] if 'FLAGS' in data.columns.names else np.full_like(magnitudes, np.nan)
    rflags = data['RFLAGS'] if 'RFLAGS' in data.columns.names else np.full_like(magnitudes, np.nan)
    julian_dates = data['JD'] if 'JD' in data.columns.names else np.full(magnitudes.shape[1], np.nan)  # Assuming JD is a 1D array with exposure length

    structured_data = []
    for i in range(magnitudes.shape[0]):  # Iterate over objects
        for j in range(magnitudes.shape[1]):  # Iterate over exposures
            obj = {
                'RA': ra_values[i],
                'Dec': dec_values[i],
                'Magnitude': magnitudes[i, j],
                'Error': errors[i, j],
                'Flags': flags[i, j],
                'RFlags': rflags[i, j],
                'Julian Date': julian_dates[j] if julian_dates.shape[0] > j else np.nan
            }
            structured_data.append(obj)
    return structured_data

# Open the FITS file
with fits.open(file_path) as hdul:
    # Print a summary of the contents of the FITS file
    hdul.info()

    # Access and print key information of the primary HDU header
    primary_hdu = hdul[0]
    #print("Primary HDU Header:")
    #print_specific_header_info(primary_hdu.header)
    #print("=" * 40)

    # Access the data (assuming data is in a table in the first extension HDU)
    if len(hdul) > 1:
        data_hdu = hdul[1]
        data = data_hdu.data

        # Print available columns in the data table
        print("Available columns:", data.columns.names)
        print("=" * 40)

        # Extract and print Min/Max RA and Dec
        get_min_max_ra_dec(data)
        print("=" * 40)

        # Count and print number of exposures and objects
        num_exposures, num_objects = count_exposures_objects(data)
        print("=" * 40)

        # Create and print the data structure
        structured_data = create_data_structure(data)
        #for obj in structured_data:
            #print(obj)
    else:
        print("No extension HDU found with data.")

In [ ]:
# Function to print specific header information
def print_specific_header_info(header):
    key_info = ['SIMPLE', 'BITPIX', 'NAXIS', 'EXTEND', 'DATE', 'ORIGIN', 'TELESCOP', 'INSTRUME', 'OBJECT']
    for key in key_info:
        if key in header:
            print(f"{key}: {header[key]}")
        else:
            print(f"{key}: Key not found")

# Function to create lists of RA and Dec
def create_ra_dec_lists(data):
    ra_list = list(data['RA'])
    dec_list = list(data['DEC'])
    return ra_list, dec_list

# Function to convert data to native endianness if necessary
def convert_to_native_endianness(data):
    if data.dtype.byteorder not in ('=', '|'):
        data = data.byteswap().newbyteorder()
    return data

# Function to reshape 3D array into 2D DataFrame
def reshape_3d_to_dataframe(data, col_name):
    if col_name in data.columns.names:
        arr = convert_to_native_endianness(data[col_name])[0]  # Remove the singleton dimension
        num_objects, num_exposures = arr.shape
        
        # Create DataFrame
        df = pd.DataFrame(arr, columns=[f'{col_name}{i+1}' for i in range(num_exposures)], index=[f'ID {i+1}' for i in range(num_objects)])
        
        return df
    else:
        print(f"{col_name} column not found in the data.")
        return pd.DataFrame()

# Function to print information for a specific object and exposure
def print_object_exposure_info(object_id, exposure, ra_list, dec_list, df_magnitude, df_errors, df_flags, df_rflags, julian_dates):
    if object_id < 1 or object_id > len(ra_list):
        print("Invalid object ID")
        return
    
    if exposure < 1 or exposure > df_magnitude.shape[1]:
        print("Invalid exposure number")
        return
    
    ra = ra_list[object_id - 1]
    dec = dec_list[object_id - 1]
    magnitude = df_magnitude.iloc[object_id - 1, exposure - 1]
    error = df_errors.iloc[object_id - 1, exposure - 1]
    flags = df_flags.iloc[object_id - 1, exposure - 1]
    rflags = df_rflags.iloc[object_id - 1, exposure - 1]
    julian_date = julian_dates[exposure - 1]
    
    print(f"Object ID: {object_id}")
    print(f"Exposure: {exposure}")
    print(f"RA: {ra}")
    print(f"Dec: {dec}")
    print(f"Magnitude: {magnitude}")
    print(f"Error in Magnitude: {error}")
    print(f"FLAGS: {flags}")
    print(f"RFLAGS: {rflags}")
    print(f"Julian Date: {julian_date}")
    print("=" * 40)

# Open the FITS file and process the data
with fits.open(file_path) as hdul:
    # Print a summary of the contents of the FITS file
    #hdul.info()

    # Access and print key information of the primary HDU header
    primary_hdu = hdul[0]
    #print("Primary HDU Header:")
    #print_specific_header_info(primary_hdu.header)
    #print("=" * 40)

    # Access the data (assuming data is in a table in the first extension HDU)
    if len(hdul) > 1:
        data_hdu = hdul[1]
        data = data_hdu.data

        # Print available columns in the data table
        print("Available columns:", data.columns.names)
        print("=" * 40)

        # Create lists of RA and Dec
        ra_list, dec_list = create_ra_dec_lists(data)
        
        # Print Lists of RA and Dec
        print("List 1 (RA):")
        for i, ra in enumerate(ra_list):
            print(f"ID {i+1} : {ra}")

        print("\nList 2 (Dec):")
        for i, dec in enumerate(dec_list):
            print(f"ID {i+1} : {dec}")
        print("=" * 40)

        # Create DataFrames for Magnitudes (M), Errors (MERR), and Flags (FLAGS, RFLAGS)
        df_magnitude = reshape_3d_to_dataframe(data, 'M')
        df_errors = reshape_3d_to_dataframe(data, 'MERR')
        df_flags = reshape_3d_to_dataframe(data, 'FLAGS')
        df_rflags = reshape_3d_to_dataframe(data, 'RFLAGS')
        
        # Print DataFrames
        #print("DataFrame 1 (Magnitudes):")
        #print(df_magnitude)
        #print("=" * 40)

        #print("DataFrame 2 (Errors in Magnitudes):")
        #print(df_errors)
        #print("=" * 40)

        #print("DataFrame 3 (Flags):")
        #print(df_flags)
        #print("=" * 40)

        #print("DataFrame 4 (RFlags):")
        #print(df_rflags)
        #print("=" * 40)

        # Extract Julian Dates
        julian_dates = list(data['JD'][0]) if 'JD' in data.columns.names else [np.nan] * df_magnitude.shape[1]

        # Example of printing information for a specific object and exposure
        object_id = 1  # Change to desired object ID
        exposure = 1   # Change to desired exposure number
        print_object_exposure_info(object_id, exposure, ra_list, dec_list, df_magnitude, df_errors, df_flags, df_rflags, julian_dates)
    else:
        print("No extension HDU found with data.")

In [ ]:
def print_object_exposure_info(object_index, exposure_index, ra_list, dec_list, df_magnitude, df_errors, df_flags, df_rflags, julian_dates):
    try:
        # Ensure we are working with numpy arrays and flatten them
        ra_list = np.asarray(ra_list).flatten()
        dec_list = np.asarray(dec_list).flatten()
        
        # Check bounds
        if object_index >= ra_list.shape[0] or object_index >= dec_list.shape[0]:
            raise IndexError("Object index is out of bounds for RA/DEC list")
        if exposure_index >= df_magnitude.shape[1]:
            raise IndexError("Exposure index is out of bounds for Magnitude DataFrame")
        if object_index >= df_magnitude.shape[0]:
            raise IndexError("Object index is out of bounds for Magnitude DataFrame")
        if exposure_index >= len(julian_dates):
            raise IndexError("Exposure index is out of bounds for Julian Dates")

        # Get the RA and Dec values for the object
        ra_value = ra_list[object_index]
        dec_value = dec_list[object_index]

        # Get the magnitude, error, and flags for the object at the given exposure
        magnitude = df_magnitude.iloc[object_index, exposure_index]
        error = df_errors.iloc[object_index, exposure_index]
        flag = df_flags.iloc[object_index, exposure_index]
        rflag = df_rflags.iloc[object_index, exposure_index]
        julian_date = julian_dates[exposure_index]

        # Print the information
        print(f"RA: {ra_value}")
        print(f"Dec: {dec_value}")
        print(f"Magnitude: {magnitude}")
        print(f"Error: {error}")
        print(f"Flag: {flag}")
        print(f"RFlag: {rflag}")
        print(f"Julian Date: {julian_date}")
    except IndexError as e:
        print(f"IndexError: {e}")
    except KeyError as e:
        print(f"KeyError: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [ ]:
print_object_exposure_info(10,10,ra_list, dec_list, df_magnitude, df_errors, df_flags, df_rflags, julian_dates)

In [ ]:
# Example function to extract object exposure information
def extract_object_exposure_info(object_index, ra_list, dec_list, df_magnitude, df_errors, df_flags, df_rflags):
    try:
        # Ensure we are working with numpy arrays and flatten them
        ra_list = np.asarray(ra_list).flatten()
        dec_list = np.asarray(dec_list).flatten()
        
        # Check bounds
        if object_index >= ra_list.shape[0] or object_index >= dec_list.shape[0]:
            raise IndexError("Object index is out of bounds for RA/DEC list")
        if object_index >= df_magnitude.shape[0]:
            raise IndexError("Object index is out of bounds for Magnitude DataFrame")
        
        # Get the RA and Dec values for the object
        ra_value = ra_list[object_index]
        dec_value = dec_list[object_index]

        # Get the magnitude, error, and flags for all exposures for the object
        magnitudes = df_magnitude.iloc[object_index, :].to_numpy()
        errors = df_errors.iloc[object_index, :].to_numpy()
        flags = df_flags.iloc[object_index, :].to_numpy()
        rflags = df_rflags.iloc[object_index, :].to_numpy()
        
        # Set magnitudes greater than 25 and negative errors to NaN
        magnitudes[(magnitudes > 25) | (errors < 0)] = np.nan
        errors[(magnitudes > 25) | (errors < 0)] = np.nan
        
        return ra_value, dec_value, magnitudes, errors, flags, rflags
    
    except IndexError as e:
        print(f"IndexError: {e}")
        return None, None, None, None, None, None
    except KeyError as e:
        print(f"KeyError: {e}")
        return None, None, None, None, None, None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None, None, None, None, None, None

# Example usage to iterate through all exposures and store magnitudes and errors
object_index = 6704
# Assuming ra_list, dec_list, df_magnitude, df_errors, df_flags, df_rflags, and julian_dates are already defined
ra_value, dec_value, magnitudes, errors, flags, rflags = extract_object_exposure_info(object_index, ra_list, dec_list, df_magnitude, df_errors, df_flags, df_rflags)

In [ ]:

# Convert to NumPy arrays if they are not already
julian_dates = np.array(julian_dates)
magnitudes = np.array(magnitudes)
errors = np.array(errors)

# Cut the data within one Julian Date of the first exposure
first_julian_date = julian_dates[0]
cutoff_julian_date = first_julian_date + 1.0

# Apply the cut
mask = julian_dates <= cutoff_julian_date
julian_dates_cut = julian_dates[mask]
magnitudes_cut = magnitudes[mask]
errors_cut = errors[mask]

# Plotting the data
plt.figure(figsize=(18, 10))
plt.errorbar(julian_dates_cut, magnitudes_cut, yerr=errors_cut, fmt='o', color='k', ecolor='r', capsize=5)
plt.xlabel('Julian Date', fontsize=10)
plt.ylabel('Magnitude', fontsize=10)
plt.title(f'Magnitude with Errors for RA: {ra_value:.6f}, Dec: {dec_value:.6f}', fontsize=14)

plt.gca().invert_yaxis()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
def match_object_by_ra_dec(ra_input, dec_input, radius, ra_list, dec_list):
    try:
        # Ensure RA and Dec lists are numpy arrays
        ra_list = np.asarray(ra_list).flatten()
        dec_list = np.asarray(dec_list).flatten()
        
        # Convert input RA/Dec and object RA/Dec to SkyCoord objects
        input_coord = SkyCoord(ra_input * u.deg, dec_input * u.deg, frame='icrs')
        obj_coords = SkyCoord(ra_list * u.deg, dec_list * u.deg, frame='icrs')
        
        # Compute separations
        separations = input_coord.separation(obj_coords)
        
        # Find objects within the specified radius
        within_radius = separations <= radius * u.deg
        
        # If no objects found, return None
        if not np.any(within_radius):
            print("No objects found within the specified radius.")
            return None
        
        # Get the index of the closest object within the radius
        matched_index = np.where(within_radius)[0][0]
        
        # Return the matched index (ID)
        return matched_index
    
    except IndexError as e:
        print(f"IndexError: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [ ]:
# Example usage to iterate through all exposures and store magnitudes and errors
ra_input = 356.383  # Example RA
dec_input = 30.459 # Example Dec
radius = 0.01  # Example radius in degrees

# Assuming ra_list and dec_list are already defined
matched_index = match_object_by_ra_dec(ra_input, dec_input, radius, ra_list, dec_list)

if matched_index is not None:
    print(f"Matched object ID (Index): {matched_index}")
else:
    print("Could not find a matching object.")

In [ ]:
##### Assuming ra_list, dec_list, df_magnitude, df_errors, df_flags, df_rflags, and julian_dates are already defined
ra_value, dec_value, magnitudes, errors, flags, rflags = extract_object_exposure_info(matched_index, ra_list, dec_list, df_magnitude, df_errors, df_flags, df_rflags)

In [ ]:
# Convert to NumPy arrays if they are not already
julian_dates = np.array(julian_dates)
magnitudes = np.array(magnitudes)
errors = np.array(errors)

# Cut the data within one Julian Date of the first exposure
first_julian_date = julian_dates[0]
cutoff_julian_date = first_julian_date + 1.0

# Apply the cut
mask = julian_dates <= cutoff_julian_date
julian_dates_cut = julian_dates[mask]
magnitudes_cut = magnitudes[mask]
errors_cut = errors[mask]

plt.figure(figsize=(18, 10))
plt.errorbar(julian_dates_cut, magnitudes_cut, yerr=errors_cut, fmt='o', color='k', ecolor='r', capsize=5)

# Labels and title
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('Magnitude', fontsize=20)
plt.title(f'Magnitude with Errors for RA: {ra_value:.6f}, Dec: {dec_value:.6f}', fontsize=20)

# Assuming magnitudes_cut is a numpy array
magnitudes_cut = magnitudes_cut[~np.isnan(magnitudes_cut)]
mean_magnitude = np.mean(magnitudes_cut)
median_magnitude = np.median(magnitudes_cut)
std_magnitude = np.std(magnitudes_cut)
print('Mean: ',mean_magnitude)
print('Median: ', median_magnitude)
print('Std:', std_magnitude)

# Invert y-axis and show grid
plt.gca().invert_yaxis()
plt.grid(True)
plt.tight_layout()
plt.show()